## Imports & Connection

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

In [2]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [3]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)

local_datasets = '/Users/rapido/local-datasets/customer/appography/demographic_personas/'
print(local_datasets)

data_dumper_path = '/Users/rapido/local-datasets/customer/appography/appography_data_dump/'

/Users/rapido/analytics/customer/Appography/demographic_personas
/Users/rapido/local-datasets/customer/appography/demographic_personas/


## Parameter & Datasets

In [4]:
start_date = '20240301' 
end_date = '20240331'

### City Base Customer

- City-wise android fare estimate customers taked from the recent month to built the datasets.

- Criteria 

        - Andrioid Customer
        - Minimum 1 FE in recent month

fe_customer_query = f"""        
        
        WITH fe_customer AS (

        SELECT
            user_id as user_id
        FROM
            hive.pricing.fare_estimates_enriched
        WHERE 
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND platform = 'android'
        GROUP BY 1
        ),

        demographic_data AS ( 

        SELECT
            customerId,
            CASE WHEN age BETWEEN 15 AND 80 THEN age END age,
            gender
        FROM 
            datasets.customer_single_view
        )

        SELECT
            user_id,
            age,
            gender
        FROM 
            fe_customer
        LEFT JOIN 
            demographic_data
            on user_id = customerId
"""

df_fe_customer = pd.read_sql(fe_customer_query, connection)
df_fe_customer.to_csv(local_datasets + 'raw/pan_fe_customer_{}_{}.csv'.format(start_date, end_date), index=False)

In [5]:
df_fe_customer = pd.read_csv(local_datasets + 'raw/pan_fe_customer_{}_{}.csv'.format(start_date, end_date))

In [6]:
df_fe_customer

,user_id,age,gender
0,5fed72c9bd3aea15ec38af3e,43.0,MALE
1,642ec3e67f62a44b28c6aada,NaN,MALE
2,641a415c233e62a841e45f66,NaN,MALE
3,58d290aa94c7205e2197fc08,17.0,MALE
4,64c6687a0cc88f112c74fee6,NaN,MALE
...,...,...,...
15451600,61a72aa717dd6a575806abc8,NaN,FEMALE
15451601,65a9f5d9a008e8863884574b,NaN,FEMALE
15451602,62f2328c3871cf30870f39eb,NaN,MALE
15451603,62d10613852b2118f29420b2,NaN,MALE


### Base Customers Recent Appograpgy Data

- Read the local appography data dumper and filter the data by selected city customers

In [7]:
start_time = time.time()

csv_files = [file for file in os.listdir(data_dumper_path) if file.endswith('.csv')]

dfs = []

# Loop through each CSV file, read it into a DataFrame, and append to the list
for file in csv_files:
    file_path = os.path.join(data_dumper_path, file)
    df = pd.read_csv(file_path)
    dfs.append(df)

# Concatenate all DataFrames in the list into a single DataFrame
df_combined_data_dumper = pd.concat(dfs, ignore_index=True)
end_time = time.time()
execution_time = ((end_time - start_time)/60.00)
print("Execution time : ", execution_time, " mins")
df_combined_data_dumper.head(5)

Execution time :  2.5500680526097614  mins


,last_sync_date,user_id,app_list_set
0,20240324,573f28f69b0ffc283676ff35,"['axis mobile', 'microsoft teams', 'whatsapp',..."
1,20240324,573f28f69b0ffc283676ff57,"['whatsapp', 'google photos', 'truecaller', 'a..."
2,20240324,573f28f79b0ffc2836770a38,"['onedrive', 'youtube', 'telegram', 'jio cinem..."
3,20240324,573f28f89b0ffc2836770daf,"['zomato', 'gmail', 'blusmart', 'google maps',..."
4,20240324,573f28f99b0ffc2836771403,"['cleartrip', 'google photos', 'linkedin', 'go..."


#### Optimization & work-around

In [8]:
df_combined_data_dumper.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31723947 entries, 0 to 31723946
Data columns (total 3 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   last_sync_date  int64 
 1   user_id         object
 2   app_list_set    object
dtypes: int64(1), object(2)
memory usage: 726.1+ MB


In [9]:
df_combined_data_dumper['last_sync_date'] = df_combined_data_dumper['last_sync_date'].astype('int32')
df_combined_data_dumper.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31723947 entries, 0 to 31723946
Data columns (total 3 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   last_sync_date  int32 
 1   user_id         object
 2   app_list_set    object
dtypes: int32(1), object(2)
memory usage: 605.1+ MB


In [10]:
## Latest appography sync data
df_combined_data_dumper = df_combined_data_dumper.sort_values(by=['user_id', 'last_sync_date'], ascending=[True, False])
df_combined_data_dumper = df_combined_data_dumper.drop_duplicates(subset='user_id', keep='first')
df_combined_data_dumper.rename(columns = {'user_id' : 'customer_id'}, inplace = True)
df_combined_data_dumper = df_combined_data_dumper[['customer_id', 'app_list_set']]
df_combined_data_dumper.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17829492 entries, 25248151 to 18082310
Data columns (total 2 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   customer_id   object
 1   app_list_set  object
dtypes: object(2)
memory usage: 408.1+ MB


#### Merge 

In [11]:
df_fe_customer.head(2)

,user_id,age,gender
0,5fed72c9bd3aea15ec38af3e,43.0,MALE
1,642ec3e67f62a44b28c6aada,NaN,MALE


In [12]:
df_combined_data_dumper.head(2)

,customer_id,app_list_set
25248151,5737c6acddbec2203f73315e,"['flipkart', 'disney+ hotstar', 'messenger', '..."
23953873,5737c6adddbec2203f73316d,"['whatsapp', 'paytm money', 'samsung galaxy we..."


In [13]:
## Merge City Base Customer & Latest appography sync data
df_base_customers_app_data = pd.merge(df_fe_customer,
                                      df_combined_data_dumper,
                                      how = 'inner',
                                      left_on = ['user_id'],
                                      right_on = ['customer_id']
                                     )
df_base_customers_app_data.head(1)

,user_id,age,gender,customer_id,app_list_set
0,5fed72c9bd3aea15ec38af3e,43.0,MALE,5fed72c9bd3aea15ec38af3e,"['paytm money', 'facebook', 'netflix', 'youtub..."


In [14]:
df_base_customers_app_data[['user_id', 'age', 'gender', 'app_list_set']]\
.to_csv(local_datasets + 'raw/pan_fe_customer_with_appo_data.csv', index=False)

In [15]:
df_base_customers_app_data = pd.read_csv(local_datasets + 'raw/pan_fe_customer_with_appo_data.csv')

In [17]:
df_base_customers_app_data

,user_id,age,gender,app_list_set
0,5fed72c9bd3aea15ec38af3e,43.0,MALE,"['paytm money', 'facebook', 'netflix', 'youtub..."
1,642ec3e67f62a44b28c6aada,NaN,MALE,"['google sheets', 'pharmeasy', 'udemy', 'linke..."
2,641a415c233e62a841e45f66,NaN,MALE,"['clock', 'google maps', 'flipkart', 'youtube'..."
3,58d290aa94c7205e2197fc08,17.0,MALE,"['google calendar', 'whatsapp', 'cleartrip', '..."
4,64c6687a0cc88f112c74fee6,NaN,MALE,"['microsoft 365', 'samsung galaxy wearable', '..."
...,...,...,...,...
13308152,65f79347651791776dc3679e,NaN,FEMALE,"['groww', 'paytm money', 'zomato', 'flipkart',..."
13308153,61a72aa717dd6a575806abc8,NaN,FEMALE,"['wynk music', 'youtube', 'google maps', 'payt..."
13308154,65a9f5d9a008e8863884574b,NaN,FEMALE,"['clock', 'google photos', 'ola', 'sonyliv', '..."
13308155,62f2328c3871cf30870f39eb,NaN,MALE,"['netflix', 'swiggy', 'zomato', 'youtube', 'te..."


In [18]:
df_base_customers_app_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13308157 entries, 0 to 13308156
Data columns (total 4 columns):
 #   Column        Dtype  
---  ------        -----  
 0   user_id       object 
 1   age           float64
 2   gender        object 
 3   app_list_set  object 
dtypes: float64(1), object(3)
memory usage: 406.1+ MB


In [20]:
df_base_customers_app_data['age'] = df_base_customers_app_data['age'].fillna(0)
df_base_customers_app_data['age'] = df_base_customers_app_data['age'].astype('int32')
df_base_customers_app_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13308157 entries, 0 to 13308156
Data columns (total 4 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   user_id       object
 1   age           int32 
 2   gender        object
 3   app_list_set  object
dtypes: int32(1), object(3)
memory usage: 355.4+ MB


In [21]:
df_base_customers_app_data

,user_id,age,gender,app_list_set
0,5fed72c9bd3aea15ec38af3e,43,MALE,"['paytm money', 'facebook', 'netflix', 'youtub..."
1,642ec3e67f62a44b28c6aada,0,MALE,"['google sheets', 'pharmeasy', 'udemy', 'linke..."
2,641a415c233e62a841e45f66,0,MALE,"['clock', 'google maps', 'flipkart', 'youtube'..."
3,58d290aa94c7205e2197fc08,17,MALE,"['google calendar', 'whatsapp', 'cleartrip', '..."
4,64c6687a0cc88f112c74fee6,0,MALE,"['microsoft 365', 'samsung galaxy wearable', '..."
...,...,...,...,...
13308152,65f79347651791776dc3679e,0,FEMALE,"['groww', 'paytm money', 'zomato', 'flipkart',..."
13308153,61a72aa717dd6a575806abc8,0,FEMALE,"['wynk music', 'youtube', 'google maps', 'payt..."
13308154,65a9f5d9a008e8863884574b,0,FEMALE,"['clock', 'google photos', 'ola', 'sonyliv', '..."
13308155,62f2328c3871cf30870f39eb,0,MALE,"['netflix', 'swiggy', 'zomato', 'youtube', 'te..."


### Base Customers Recent Use-Case Data

In [30]:
customer_use_case_query = f"""

    WITH use_case AS (

        SELECT
            hex_10, 
            usecase_tag
        FROM
            (
            SELECT 
                hex_10, 
                combined_final_usecase_accuracy as usecase_tag,
                ROW_NUMBER() OVER(PARTITION BY hex_10 ORDER BY run_date DESC) seq_no
            FROM
                hive.experiments_internal.combined_usecase_hex_10_level
            )
        WHERE   
            seq_no = 1
        ),

        active_customers AS (

        SELECT 
            customer_id,
            order_id,
            drop_location_hex_10 hex_10,
            COALESCE(usecase_tag, 'Unknown') usecase_tag,
            'first_half_drop' time_period
        FROM 
            orders.order_logs_snapshot
        LEFT JOIN 
            use_case a 
            ON drop_location_hex_10 = hex_10
        WHERE
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND channel_host = 'android'
            AND cast(substr(quarter_hour,1,2) as real) >= 8 
            AND cast(substr(quarter_hour,1,2) as real) <= 12    
        GROUP BY 1,2,3,4

        UNION 

            SELECT 
            customer_id,
            order_id,
            pickup_location_hex_10 hex_10,
            COALESCE(usecase_tag, 'Unknown') usecase_tag,
            'second_half_pickup' time_period
        FROM 
            orders.order_logs_snapshot
        LEFT JOIN 
            use_case a 
            ON pickup_location_hex_10 = hex_10
        WHERE
            yyyymmdd >= '20240401'
            AND yyyymmdd <= '20240413'
            AND channel_host = 'android'
            AND cast(substr(quarter_hour,1,2) as real) >= 13
            AND cast(substr(quarter_hour,1,2) as real) <= 21    
        GROUP BY 1,2,3,4

        ),

        merge AS (

        SELECT
            customer_id,
            time_period,
            usecase_tag,
            orders,
            ROW_NUMBER() OVER(PARTITION BY customer_id,time_period ORDER BY orders DESC) seq_no
        FROM 
            (
            SELECT
                customer_id,
                time_period,
                usecase_tag,
                COUNT(DISTINCT order_id) orders
            FROM 
                active_customers
            GROUP BY 1,2,3
            )
        -- WHERE 
        --     LOWER(usecase_tag) != 'unknown'
        )

        SELECT
            customer_id,
            COALESCE(first_half_drop_primary, 'NA') first_half_drop_primary,
            COALESCE(first_half_drop_secondary, 'NA') first_half_drop_secondary,
            COALESCE(second_half_pickup_primary, 'NA') second_half_pickup_primary,
            COALESCE(second_half_pickup_secondary, 'NA') second_half_pickup_secondary
        FROM
        (
        SELECT
            customer_id,
            MAX(CASE WHEN time_period = 'first_half_drop' AND seq_no = 1 THEN usecase_tag END) first_half_drop_primary,
            MAX(CASE WHEN time_period = 'first_half_drop' AND seq_no = 2 THEN usecase_tag END) first_half_drop_secondary,
            MAX(CASE WHEN time_period = 'second_half_pickup' AND seq_no = 1 THEN usecase_tag END) second_half_pickup_primary,
            MAX(CASE WHEN time_period = 'second_half_pickup' AND seq_no = 2 THEN usecase_tag END) second_half_pickup_secondary
        FROM 
            merge 
        WHERE 
            seq_no in (1,2)
        GROUP BY 1
        )
"""

df_customer_use_case = pd.read_sql(customer_use_case_query, connection) 
df_customer_use_case.to_csv(local_datasets + 'raw/pan_customer_use_case_{}_{}.csv'.format(start_date, end_date), index=False)

In [31]:
df_customer_use_case = pd.read_csv(local_datasets + 'raw/pan_customer_use_case_{}_{}.csv'.format(start_date, end_date))
df_customer_use_case.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9081027 entries, 0 to 9081026
Data columns (total 5 columns):
 #   Column                        Dtype 
---  ------                        ----- 
 0   customer_id                   object
 1   first_half_drop_primary       object
 2   first_half_drop_secondary     object
 3   second_half_pickup_primary    object
 4   second_half_pickup_secondary  object
dtypes: object(5)
memory usage: 346.4+ MB


#### Merge

In [32]:
## Merge City Base Customer & Latest appography sync data
df_customers_data = pd.merge(df_base_customers_app_data,
                             df_customer_use_case,
                             how = 'left',
                             left_on = ['user_id'],
                             right_on = ['customer_id']
                             )
df_customers_data.head(1)

,user_id,age,gender,app_list_set,customer_id,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary
0,5fed72c9bd3aea15ec38af3e,43,MALE,"['paytm money', 'facebook', 'netflix', 'youtub...",5fed72c9bd3aea15ec38af3e,residential,Unknown,health_and_personal,NaN


In [33]:
df_customers_data[['user_id', 'age', 'gender',
                   'first_half_drop_primary', 'first_half_drop_secondary', 
                   'second_half_pickup_primary', 'second_half_pickup_secondary', 'app_list_set']]\
.to_csv(local_datasets + 'processed/pan_customer_use_case_{}_{}.csv'.format(start_date, end_date), index=False)

### Final data

In [35]:
df_customers_data[['user_id', 'age', 'gender',
                   'first_half_drop_primary', 'first_half_drop_secondary', 
                   'second_half_pickup_primary', 'second_half_pickup_secondary', 'app_list_set']].head(5)

,user_id,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list_set
0,5fed72c9bd3aea15ec38af3e,43,MALE,residential,Unknown,health_and_personal,NaN,"['paytm money', 'facebook', 'netflix', 'youtub..."
1,642ec3e67f62a44b28c6aada,0,MALE,Unknown,residential,office,NaN,"['google sheets', 'pharmeasy', 'udemy', 'linke..."
2,641a415c233e62a841e45f66,0,MALE,residential,transit_station,NaN,NaN,"['clock', 'google maps', 'flipkart', 'youtube'..."
3,58d290aa94c7205e2197fc08,17,MALE,residential,transit_station,NaN,NaN,"['google calendar', 'whatsapp', 'cleartrip', '..."
4,64c6687a0cc88f112c74fee6,0,MALE,NaN,NaN,NaN,NaN,"['microsoft 365', 'samsung galaxy wearable', '..."


In [36]:
df_customers_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13308157 entries, 0 to 13308156
Data columns (total 9 columns):
 #   Column                        Dtype 
---  ------                        ----- 
 0   user_id                       object
 1   age                           int32 
 2   gender                        object
 3   app_list_set                  object
 4   customer_id                   object
 5   first_half_drop_primary       object
 6   first_half_drop_secondary     object
 7   second_half_pickup_primary    object
 8   second_half_pickup_secondary  object
dtypes: int32(1), object(8)
memory usage: 863.0+ MB


In [39]:
df_customers_data.columns

Index(['user_id', 'age', 'gender', 'app_list_set', 'customer_id',
       'first_half_drop_primary', 'first_half_drop_secondary',
       'second_half_pickup_primary', 'second_half_pickup_secondary'],
      dtype='object')